# Distracted Driver Detection — Colab Training

**Runtime → Change runtime type → T4 GPU**

This notebook:
1. Clones your repo from GitHub
2. Downloads the State Farm dataset via Kaggle API
3. Trains MobileNetV2 with 2-phase warm-up strategy
4. Exports the best model to ONNX
5. Saves weights to Google Drive

In [ ]:
# ── 1. Check GPU ──────────────────────────────────────────────────────────────
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT AVAILABLE')
print('CUDA:', torch.version.cuda)
!nvidia-smi | head -20

In [ ]:
# ── 2. Clone your repo ────────────────────────────────────────────────────────
# Replace with your actual GitHub repo URL after you push
REPO_URL = 'https://github.com/YOUR_USERNAME/distracted-driver-detection-edge.git'

!git clone {REPO_URL} /content/project
%cd /content/project

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────────
!pip install -q -r requirements.txt

In [ ]:
# ── 4. Download dataset from Kaggle ──────────────────────────────────────────
# Option A: Upload your kaggle.json API key
from google.colab import files
print('Upload your kaggle.json (from https://www.kaggle.com/settings → API → Create New Token)')
uploaded = files.upload()  # select kaggle.json

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download State Farm dataset
!kaggle competitions download -c state-farm-distracted-driver-detection -p /content/project/data/raw/
!cd /content/project/data/raw && unzip -q state-farm-distracted-driver-detection.zip
!ls /content/project/data/raw/imgs/train/ | head

In [ ]:
# ── 4B. Alternative: Mount Google Drive (if you pre-uploaded the dataset) ────
# from google.colab import drive
# drive.mount('/content/drive')
# DATASET_PATH = '/content/drive/MyDrive/datasets/state-farm'
# !cp -r {DATASET_PATH}/imgs /content/project/data/raw/

In [ ]:
# ── 5. Verify dataset ─────────────────────────────────────────────────────────
import os
train_dir = '/content/project/data/raw/imgs/train'
for cls in sorted(os.listdir(train_dir)):
    n = len(os.listdir(os.path.join(train_dir, cls)))
    print(f'  {cls}: {n} images')

In [ ]:
# ── 6. Train ──────────────────────────────────────────────────────────────────
!python scripts/train.py \
    --epochs 20 \
    --warmup 5 \
    --batch-size 64 \
    --device cuda \
    --workers 4

In [ ]:
# ── 7. Export to ONNX ─────────────────────────────────────────────────────────
!python scripts/export_model.py \
    --weights models/weights/best_model.pt \
    --output  models/weights/driver_classifier.onnx \
    --verify

In [ ]:
# ── 8. Show training history ──────────────────────────────────────────────────
import json, matplotlib.pyplot as plt

with open('models/weights/training_history.json') as f:
    history = json.load(f)

epochs     = [h['epoch'] for h in history]
train_acc  = [h['train_acc'] for h in history]
val_acc    = [h['val_acc'] for h in history]
train_loss = [h['train_loss'] for h in history]
val_loss   = [h['val_loss'] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_acc, label='Train'); ax1.plot(epochs, val_acc, label='Val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.plot(epochs, train_loss, label='Train'); ax2.plot(epochs, val_loss, label='Val')
ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

best = max(history, key=lambda h: h['val_acc'])
print(f"Best → epoch {best['epoch']}  val_acc={best['val_acc']:.4f}  val_loss={best['val_loss']:.4f}")

In [ ]:
# ── 9. Save weights to Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/distracted-driver-weights'
!mkdir -p {DRIVE_SAVE_DIR}
!cp models/weights/best_model.pt           {DRIVE_SAVE_DIR}/
!cp models/weights/driver_classifier.onnx  {DRIVE_SAVE_DIR}/
!cp models/weights/training_history.json   {DRIVE_SAVE_DIR}/
print('Saved to Google Drive:', DRIVE_SAVE_DIR)

In [ ]:
# ── 10. (Optional) Download weights directly to your machine ─────────────────
from google.colab import files
files.download('models/weights/best_model.pt')
files.download('models/weights/driver_classifier.onnx')